# PyTorch: the recorded graph

Sections 3.1 and 15.1 of the notes.

PyTorch builds the computational graph of Chapter 3 while the program runs.
This notebook takes the example of Chapter 3, reads the graph that PyTorch
built, and checks the reverse sweep against the derivatives computed by hand.

In [ ]:
import torch

torch.set_default_dtype(torch.float64)

x1 = torch.tensor(2.0, requires_grad=True)
x2 = torch.tensor(3.0, requires_grad=True)

v3 = x1 * x2
v4 = x2**3
v5 = torch.sin(v3)
y = v4 + v5  # the graph of Figure 3.2

print("y        ", y.item())
print("grad_fn  ", type(y.grad_fn).__name__)
print("leaf?    ", x1.is_leaf, y.is_leaf)

## 1. Reading the graph

Each node knows the function that produced it and the nodes it came from.
Walking `next_functions` prints the graph of Definition 3.1 backwards, from the
output to the inputs.

In [ ]:
def show(fn, depth=0, seen=None):
    if fn is None:
        return
    print("  " * depth + type(fn).__name__)
    for parent, _ in getattr(fn, "next_functions", ()):
        show(parent, depth + 1)


show(y.grad_fn)

## 2. The reverse sweep

`backward` runs Proposition 3.5 with the seed 1 and stores the result in the
`grad` attribute of the input nodes. The values are those of Example 3.7.

In [ ]:
y.backward()
print("d/dx1", x1.grad.item(), " expected", (x2 * torch.cos(x1 * x2)).item())
print(
    "d/dx2", x2.grad.item(), " expected", (x1 * torch.cos(x1 * x2) + 3 * x2**2).item()
)

## 3. Gradients accumulate

A second sweep adds to what is already stored. Training loops therefore clear
the gradients before each step, and forgetting to do so is a common mistake.

In [ ]:
y = torch.sin(x1 * x2) + x2**3
y.backward()
print("after a second sweep:", x1.grad.item())

x1.grad = None
x2.grad = None
y = torch.sin(x1 * x2) + x2**3
y.backward()
print("after clearing:      ", x1.grad.item())

## 4. The graph is discarded, unless it is kept

After the sweep the graph is freed. A second sweep over the same graph needs
`retain_graph=True`, and differentiating the sweep itself needs
`create_graph=True`. The second is how a second derivative is obtained.

In [ ]:
x = torch.tensor(1.0, requires_grad=True)
u = torch.sin(x**2)

(du_dx,) = torch.autograd.grad(u, x, create_graph=True)  # keeps the sweep in the graph
(d2u_dx2,) = torch.autograd.grad(du_dx, x)

print("u'  ", du_dx.item(), " expected", (2 * x * torch.cos(x**2)).item())
print(
    "u'' ",
    d2u_dx2.item(),
    " expected",
    (2 * torch.cos(x**2) - 4 * x**2 * torch.sin(x**2)).item(),
)

## 5. The residual of a physics-informed network

The same pattern gives the derivatives of a network with respect to its input,
which is what the residual of Chapter 5 needs.

In [ ]:
import torch.nn as nn

net = nn.Sequential(nn.Linear(1, 16), nn.Tanh(), nn.Linear(16, 1))

x = torch.linspace(-1, 1, 5).reshape(-1, 1).requires_grad_(True)
u = net(x)

(u_x,) = torch.autograd.grad(u, x, torch.ones_like(u), create_graph=True)
(u_xx,) = torch.autograd.grad(u_x, x, torch.ones_like(u_x), create_graph=True)

D = 2e-5
residual = D * u_xx + u - u**3  # Allen-Cahn, stationary part
print("u_x  ", u_x.flatten().tolist()[:3])
print("u_xx ", u_xx.flatten().tolist()[:3])
print("residual shape", tuple(residual.shape))